# 🧹 Notebook 02: Data Cleaning & Feature Extraction
**โครงการ**: Used Car Analytics (Data Warehouse & ETL Pipeline)
**กลุ่ม**: GroupXX

---

## 📌 วัตถุประสงค์ของ Notebook นี้:
1. กำหนดฟังก์ชันทำความสะอาดข้อมูล (Clean Functions) สกัดตัวเลขราคา ไมล์ และข้อความชื่อรถ
2. ทำความสะอาดข้อมูลดิบแยกตาม **3 Data Sources หลัก** (Kaidee Auto JSON, One2car Multi-file, US Sales Log)
3. สกัดฟีเจอร์สำคัญ: `model_year`, `brand`, `model`, `body_type` (Pick-up, SUV, Sedan, ฯลฯ), และ `fuel_type` ด้วย Regex

## 🛠️ Section 1: นิยามฟังก์ชันทำความสะอาดข้อมูล (Cleaning & Parsing Helper Functions)

In [1]:
import pandas as pd
import json
import glob
import re

# 1. Clean Price
def clean_price(val):
    if pd.isna(val): return None
    nums = re.sub(r'[^\d]', '', str(val))
    return float(nums) if nums != '' else None

# 2. Clean Mileage
def clean_mileage(val):
    if pd.isna(val): return None
    s = str(val).replace('กม.', '').replace(',', '').strip()
    match_range = re.search(r'(\d+)\s*-\s*(\d+)K', s, re.IGNORECASE)
    if match_range:
        low = float(match_range.group(1)) * 1000
        high = float(match_range.group(2)) * 1000
        return int((low + high) / 2)
    nums = re.sub(r'[^\d]', '', s)
    return int(nums) if nums != '' else None

# 3. Extract BodyType (Pick-up, SUV, Sedan, Hatchback, Coupe, Van)
def extract_body_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['pickup', 'cab', 'space cab', 'hi-lander', 'double cab', 'smart cab', 'revo', 'd-max', 'ranger', 'navara', 'กระบะ']):
        return 'Pick-up'
    elif any(k in text for k in ['suv', 'mu-x', 'fortuner', 'everest', 'cr-v', 'x3', 'glc', 'cross', 'hr-v', 'cx-5', 'pajero']):
        return 'SUV'
    elif any(k in text for k in ['hatchback', 'good cat', 'yaris', 'swift', '5 ประตู', 'ora']):
        return 'Hatchback'
    elif any(k in text for k in ['coupe', 'gran m sport', '220i']):
        return 'Coupe'
    elif any(k in text for k in ['van', 'caravelle', 'wagon', 'ตู้']):
        return 'Van'
    elif any(k in text for k in ['sedan', 'city', 'camry', 'altis', 'civic', 'mazda 3', 'c220', '520d', 'ซีดาน']):
        return 'Sedan'
    else:
        return 'Sedan'

# 4. Extract FuelType (Petrol, Diesel, Hybrid, EV)
def extract_fuel_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['e:hev', 'hev', 'hybrid', 'ไฮบริด']):
        return 'Hybrid'
    elif any(k in text for k in ['ora', 'good cat', 'ev', 'รถไฟฟ้า', '100%']):
        return 'EV'
    elif any(k in text for k in ['d-max', 'hilux', 'revo', 'ranger', 'navara', 'mu-x', 'fortuner', 'everest', '520d', 'c220 d', 'tdi', 'ดีเซล']):
        return 'Diesel'
    else:
        return 'Petrol'

# 5. Parse Unstructured Car Title
def parse_car_title(title, desc=''):
    if pd.isna(title): return pd.Series([2018, 'Unknown', 'General', 'Sedan', 'Petrol'])
    title_str = str(title).strip()
    year_match = re.search(r'^(20\d{2}|19\d{2})', title_str)
    year = int(year_match.group(1)) if year_match else 2018
    text_clean = re.sub(r'^(20\d{2}|19\d{2})\s*', '', title_str)
    parts = text_clean.split()
    brand = parts[0] if len(parts) > 0 else 'Unknown'
    model = parts[1] if len(parts) > 1 else 'General'
    body_type = extract_body_type(title_str, desc)
    fuel_type = extract_fuel_type(title_str, desc)
    return pd.Series([year, brand, model, body_type, fuel_type])

print('✅ นิยาม Cleaning Functions สำเร็จเรียบร้อย')

✅ นิยาม Cleaning Functions สำเร็จเรียบร้อย


## 🧹 Section 2: การทำความสะอาดข้อมูลแยกตาม Data Source
### 🟢 Data Source 1: Kaidee Auto (Nested JSON Data Source)

In [2]:
# 1. Clean Data Source 1: Kaidee Auto JSON
json_path = '../../01_Raw_Data/kaidee/kaidee_cars_detail.json'
with open(json_path, 'r', encoding='utf-8') as f:
    df_kaidee = pd.DataFrame(json.load(f))

df_kaidee_clean = df_kaidee.dropna(subset=['price']).copy()
df_kaidee_clean['price_clean'] = df_kaidee_clean['price'].apply(clean_price)
df_kaidee_clean['mileage_clean'] = df_kaidee_clean['mileage'].apply(clean_mileage)
df_kaidee_clean['model_year'] = pd.to_numeric(df_kaidee_clean['year'], errors='coerce').fillna(2018).astype(int)
df_kaidee_clean['body_type'] = df_kaidee_clean.apply(lambda r: extract_body_type(r.get('title'), r.get('description')), axis=1)
df_kaidee_clean['fuel_type'] = df_kaidee_clean.apply(lambda r: extract_fuel_type(r.get('title'), r.get('description')), axis=1)
df_kaidee_clean['transmission_clean'] = df_kaidee_clean['transmission'].map({'เกียร์อัตโนมัติ': 'Automatic', 'เกียร์ธรรมดา': 'Manual'}).fillna('Automatic')
df_kaidee_clean['location'] = df_kaidee_clean['location'].fillna('กรุงเทพมหานคร')

print(f'✅ [Data Source 1: Kaidee Auto] Cleaned สำเร็จ: {len(df_kaidee_clean):,} แถวสมบูรณ์')
df_kaidee_clean[['title', 'brand', 'model', 'model_year', 'body_type', 'fuel_type', 'price_clean', 'mileage_clean']].head(5)

✅ [Data Source 1: Kaidee Auto] Cleaned สำเร็จ: 1,443 แถวสมบูรณ์


,title,brand,model,model_year,body_type,fuel_type,price_clean,mileage_clean
0,F44 220i Gran Coupe Sport CBU ปี 22 2.0 192hp ...,BMW,Series 2,2022,Coupe,EV,777000.0,100420.0
1,Mitsubishi Triton 2.4 GT Premium Plus 4WD Doub...,Mitsubishi,Triton,2018,Pick-up,Petrol,499999.0,90000.0
2,W253 GLC250d AMG 2017 แท้ๆ ท็อปสุด option เต็ม...,Mercedes-Benz,GLC-Class,2017,SUV,EV,898000.0,211737.0
3,F30 320i Navi Lux แท้ ลงเล่ม 13 มือเดียว วิ่ง ...,BMW,Series 3,2013,Sedan,Petrol,479000.0,96600.0
4,MG VS HEV 1.5 X Two tone ปี2025 สีขาว-ดำ ไมล์ ...,MG,VS HEV,2025,Sedan,Hybrid,539999.0,11199.0


### 🔵 Data Source 2: One2car (Multi-file Web Scraped Data Series)

In [3]:
# 2. Clean Data Source 2: One2car Multi-file
def standardize_scraped_columns(df):
    rename_map = {
        'data': 'car_title',
        'data2': 'description',
        'data3': 'mileage',
        'data4': 'location',
        'data6': 'car_model',
        'data16': 'transmission'
    }
    return df.rename(columns=rename_map)

raw_one2car_files = sorted(glob.glob('../../01_Raw_Data/one2car/one2car-11-*.csv'))
df_list = [standardize_scraped_columns(pd.read_csv(f)) for f in raw_one2car_files]
df_one2car = pd.concat(df_list, ignore_index=True)

df_one2car_clean = df_one2car.dropna(subset=['price']).copy()
df_one2car_clean['price_clean'] = df_one2car_clean['price'].apply(clean_price)
df_one2car_clean['mileage_clean'] = df_one2car_clean['mileage'].apply(clean_mileage)
df_one2car_clean[['model_year', 'brand', 'model', 'body_type', 'fuel_type']] = df_one2car_clean.apply(lambda r: parse_car_title(r.get('car_title'), r.get('description')), axis=1)
df_one2car_clean = df_one2car_clean.dropna(subset=['price_clean']).copy()
df_one2car_clean['transmission_clean'] = df_one2car_clean['transmission'].map({'เกียร์อัตโนมัติ': 'Automatic', 'เกียร์ธรรมดา': 'Manual'}).fillna('Automatic')
df_one2car_clean['location'] = df_one2car_clean['location'].fillna('กรุงเทพมหานคร')

print(f'✅ [Data Source 2: One2car Multi-file] Cleaned สำเร็จ: {len(df_one2car_clean):,} แถวสมบูรณ์')
df_one2car_clean[['car_title', 'brand', 'model', 'model_year', 'body_type', 'fuel_type', 'price_clean', 'mileage_clean']].head(5)

✅ [Data Source 2: One2car Multi-file] Cleaned สำเร็จ: 3,944 แถวสมบูรณ์


,car_title,brand,model,model_year,body_type,fuel_type,price_clean,mileage_clean
0,2015 Honda City 1.5 (ปี 14-18) SV+ Sedan - SV,Honda,City,2015,Sedan,Petrol,269000.0,172500
1,2023 Honda City 1.0 (ปี 19-26) SV Sedan,Honda,City,2023,Sedan,Petrol,359000.0,32500
2,2025 BMW 220i 2.0 F44 (ปี 20-27) Gran M Sport ...,BMW,220i,2025,Coupe,Petrol,1250000.0,32500
3,2022 Toyota HILUX REVO 2.4 Double Cab Z Editio...,Toyota,HILUX,2022,Pick-up,EV,449000.0,112500
4,2015 Honda City 1.5 (ปี 14-18) SV Sedan,Honda,City,2015,Sedan,Petrol,259000.0,22500


### 🔴 Data Source 3: US Used Car Sales (Historical Transaction Log Data Source)

In [4]:
# 3. Clean Data Source 3: US Sales Log
df_us_sales = pd.read_csv('../../01_Raw_Data/us-usecar/used_car_sales.csv')
df_us_clean = df_us_sales[(df_us_sales['pricesold'] > 100) & (df_us_sales['Mileage'] > 0)].copy()
df_us_clean = df_us_clean.rename(columns={
    'pricesold': 'selling_price',
    'yearsold': 'sale_year',
    'Mileage': 'mileage',
    'Make': 'brand',
    'Model': 'model',
    'Year': 'model_year',
    'BodyType': 'body_type'
})
df_us_clean['body_type'] = df_us_clean['body_type'].fillna('Other')

print(f'✅ [Data Source 3: US Sales Log] Cleaned สำเร็จ: {len(df_us_clean):,} แถวสมบูรณ์')
df_us_clean[['brand', 'model', 'model_year', 'selling_price', 'mileage', 'body_type']].head(5)

✅ [Data Source 3: US Sales Log] Cleaned สำเร็จ: 119,034 แถวสมบูรณ์


,brand,model,model_year,selling_price,mileage,body_type
0,Ford,Mustang,1988,7500,84430,Sedan
2,Jaguar,XJS,1995,8750,55000,Convertible
3,Ford,Mustang,1968,11600,97200,Coupe
4,Porsche,911,2002,44000,40703,Coupe
5,Mercury,Montclair,1965,950,71300,Sedan
